# Partitioning Strategies for Observability Data

**Module 5: Partitioning and Bucketing for Observability Data** · [Quiz](quiz05_partition_and_bucket_choices.ipynb) · [Course README](../../README.md)

The tables that the exporter created, such as `otel_logs`, use daily partitions. Are there other partitioning strategies besides daily partitioning?

First, let's look at hourly partitioning:

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [ ]:
lab.execute(r"""
CREATE TABLE `otel_logs_for_hour_partition` (
  `timestamp` datetime(6) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200)  NULL,
  `trace_id` varchar(200)  NULL,
  `span_id` text  NULL,
  `severity_number` int  NULL,
  `severity_text` text  NULL,
  `body` text  NULL,
  `resource_attributes` variant  NULL,
  `log_attributes` variant  NULL,
  `scope_name` text  NULL,
  `scope_version` text  NULL
) ENGINE=OLAP
DUPLICATE KEY(`timestamp`, `service_name`)
PARTITION BY RANGE(`timestamp`)()
DISTRIBUTED BY HASH(`service_name`) BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "HOUR",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "240", -- hourly partitions for 10 days
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728"
);
""", title='Hourly partition table')


Use the following command to inspect the partitions. The result contains:

```text
242 rows in set (0.01 sec)
```

This indicates that the table created partitions for the previous 10 days, plus the current partition and one pre-created partition, for a total of 242 partitions.

However, this number of partitions is excessive and creates significant metadata management pressure:


In [ ]:
lab.sql(r"""
SHOW PARTITIONS FROM otel_logs_for_hour_partition;
""", title='Hourly partitions')


What happens when monthly partitioning is used instead?

In [ ]:
lab.execute(r"""
CREATE TABLE `otel_logs_for_month_partition` (
  `timestamp` datetime(6) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200)  NULL,
  `trace_id` varchar(200)  NULL,
  `span_id` text  NULL,
  `severity_number` int  NULL,
  `severity_text` text  NULL,
  `body` text  NULL,
  `resource_attributes` variant  NULL,
  `log_attributes` variant  NULL,
  `scope_name` text  NULL,
  `scope_version` text  NULL
) ENGINE=OLAP
DUPLICATE KEY(`timestamp`, `service_name`)
PARTITION BY RANGE(`timestamp`)()
DISTRIBUTED BY HASH(`service_name`) BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "MONTH",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "12", -- monthly partitions for 12 months
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728"
);
""", title='Monthly partition table')


Use the following command to inspect the partitions. The result contains:

```text
14 rows in set (0.00 sec)
```

This means that one year of history partitions was created, plus the current partition and one pre-created partition, for a total of 14 partitions.

In [ ]:
lab.sql(r"""
SHOW PARTITIONS FROM otel_logs_for_month_partition;
""", title='Monthly partitions')


However, monthly partitioning is too coarse-grained, so daily details cannot be analyzed independently. Therefore, in observability scenarios, daily partitioning remains the most reasonable choice unless there is a specific requirement.

## Dynamic partitioning and data governance

Doris supports static partitioning, dynamic partitioning, and automatic partitioning. Tables created by the OTel Doris exporter use dynamic partitioning. What should be considered when using dynamic partitioning in observability scenarios?

Consider another scenario: a very large amount of metric data is collected from the OTel Collector every day, and retaining all of it consumes a large amount of disk space. Metrics are mainly used for monitoring and alerting, and only the most recent month of data is typically required. How can partitioning support this type of data-governance requirement?

In [ ]:
lab.execute(r"""
CREATE TABLE `otel_metrics_for_data_governance` (
  `service_name` varchar(200) NULL,
  `timestamp` datetime(6) NULL,
  `service_instance_id` varchar(200) NULL,
  `metric_name` varchar(200) NULL,
  `metric_description` text NULL,
  `metric_unit` text NULL,
  `attributes` variant NULL,
  `start_time` datetime(6) NULL,
  `value` double NULL,
  `exemplars` array<struct<filtered_attributes:map<text,text>,timestamp:datetime(6),value:double,span_id:text,trace_id:text>> NULL,
  `resource_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_metric_name (`metric_name`) USING INVERTED,
  INDEX idx_metric_description (`metric_description`) USING INVERTED,
  INDEX idx_metric_unit (`metric_unit`) USING INVERTED,
  INDEX idx_attributes (`attributes`) USING INVERTED,
  INDEX idx_start_time (`start_time`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`service_name`, `timestamp`)
PARTITION BY RANGE(`timestamp`) ()
DISTRIBUTED BY RANDOM BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "DAY",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.start" = "-30",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "40",
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728"
);
""", title='Metrics table with dynamic partitions')


Pay particular attention to `"dynamic_partition.start" = "-30"`. It means that Doris retains only the most recent 30 days of data and deletes data older than 30 days.

Also pay attention to `"dynamic_partition.history_partition_num" = "40"` and `"dynamic_partition.create_history_partition" = "true"`:

- `"dynamic_partition.create_history_partition" = "true"` means that historical partitions are created when the table is created.
- `"dynamic_partition.history_partition_num" = "40"` specifies how many historical partitions to create.

In the preceding DDL, `dynamic_partition.history_partition_num` is greater than the absolute value of `dynamic_partition.start`. Although 40 historical partitions are requested, Doris retains only 30. Therefore, only 30 historical partitions exist after the table is created.

Use the following command to inspect the partitions. The result contains:

```text
32 rows in set (0.05 sec)
```

The total of 32 consists of 30 historical partitions, the current partition, and one pre-created partition.

In [ ]:
lab.sql(r"""
SHOW PARTITIONS FROM otel_metrics_for_data_governance;
""", title='Dynamic partitions for metrics')


## Automatic partitioning

If the dynamic-partition configuration is too extensive at table-creation time and you do not want to manage pre-created partitions, retained partition counts, or data-governance requirements, automatic partitioning is a suitable choice for observability workloads. Consider the metrics table again:

In [ ]:
lab.execute(r"""
CREATE TABLE `otel_metrics_for_auto_partition` (
  `service_name` varchar(200) NOT NULL,
  `timestamp` datetime(6) NOT NULL,
  `service_instance_id` varchar(200) NULL,
  `metric_name` varchar(200) NULL,
  `metric_description` text NULL,
  `metric_unit` text NULL,
  `attributes` variant NULL,
  `start_time` datetime(6) NULL,
  `value` double NULL,
  `exemplars` array<struct<filtered_attributes:map<text,text>,timestamp:datetime(6),value:double,span_id:text,trace_id:text>> NULL,
  `resource_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_metric_name (`metric_name`) USING INVERTED,
  INDEX idx_metric_description (`metric_description`) USING INVERTED,
  INDEX idx_metric_unit (`metric_unit`) USING INVERTED,
  INDEX idx_attributes (`attributes`) USING INVERTED,
  INDEX idx_start_time (`start_time`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`service_name`, `timestamp`)
AUTO PARTITION BY RANGE (date_trunc(`timestamp`, 'day')) ()
DISTRIBUTED BY RANDOM BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1"
);
""", title='Metrics table with automatic partitions')


This DDL is concise. When metrics data is written, Doris automatically creates any partitions that do not exist.

## Bucket distribution strategies

Recall the `daily_log_counts` table from Module 4 (Table Models for Observability Data); if it does not exist, run the Lab of Module 4 first. Queries on it often look up one service, so `service_name` appears in the `WHERE` clause, and its DDL uses HASH bucketing on that column. The table is small and has a single bucket, but the same choice applies to a large table with many buckets:

In [ ]:
lab.sql(r"""
SHOW CREATE TABLE daily_log_counts
""", title='Daily log count table layout')


This bucketing strategy places `service_name` values with the same hash value into the same bucket. When the `WHERE` clause contains `service_name`, the query can target a specific bucket and improve query performance.

However, the selected bucket column is not always perfectly distributed. Consider the same requirement, where `service_name` is queried frequently, but its values are affected by data skew. Suppose `service_name` has only two values, `mobile` and `PC`, with 1 billion rows for `mobile` and 1 million rows for `PC`. All 1 billion `mobile` rows are placed in the same bucket, creating a data hotspot that can impose significant performance overhead during data loading and queries.

For this scenario, use RANDOM bucketing, as shown by the logs table already created by the OTel Doris exporter:

In [ ]:
lab.sql(r"""
SHOW CREATE TABLE otel_logs
""", title='OTel log table layout')


If hotspot data is present, RANDOM bucketing can significantly improve load and query performance.

## Bucket count

The bucket count also requires careful consideration when configuring buckets. The bucket count specifies how many buckets exist under one partition and can be configured manually. In the following DDL, `DISTRIBUTED BY RANDOM BUCKETS 10` means that RANDOM bucketing is used and the bucket count is 10:

In [ ]:
lab.execute(r"""
CREATE TABLE `otel_metrics_for_bucket_number` (
  `service_name` varchar(200) NOT NULL,
  `timestamp` datetime(6) NOT NULL,
  `service_instance_id` varchar(200) NULL,
  `metric_name` varchar(200) NULL,
  `metric_description` text NULL,
  `metric_unit` text NULL,
  `attributes` variant NULL,
  `start_time` datetime(6) NULL,
  `value` double NULL,
  `exemplars` array<struct<filtered_attributes:map<text,text>,timestamp:datetime(6),value:double,span_id:text,trace_id:text>> NULL,
  `resource_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_metric_name (`metric_name`) USING INVERTED,
  INDEX idx_metric_description (`metric_description`) USING INVERTED,
  INDEX idx_metric_unit (`metric_unit`) USING INVERTED,
  INDEX idx_attributes (`attributes`) USING INVERTED,
  INDEX idx_start_time (`start_time`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`service_name`, `timestamp`)
AUTO PARTITION BY RANGE (date_trunc(`timestamp`, 'day')) ()
DISTRIBUTED BY RANDOM BUCKETS 10
PROPERTIES (
"replication_allocation" = "tag.location.default: 1"
);
""", title='Explicit bucket-count table')


If the bucket count can be configured manually, what difference does the count make? For the metrics table, suppose only one bucket is configured. All data in a partition is then stored in that single bucket. When metrics data is queried, only partitions can be pruned, and bucket-level pruning is unavailable, resulting in very low query efficiency. If too many buckets are configured, metadata management becomes difficult and places substantial pressure on the FE.

What bucket count is reasonable in observability scenarios? You can estimate it from the amount of data written each day. Suppose the logs data reaches 100 GB per day before compression and daily partitioning is used, so one partition contains approximately 100 GB. After compression, the size is approximately 30 GB. If one bucket is expected to store 1 to 10 GB after compression, the table should be configured with approximately 3 to 30 buckets.

This calculation is inconvenient because the daily data volume can vary, making manual bucket configuration difficult. Is there a simpler bucket-count strategy? The OTel Doris exporter already demonstrates one:

In [ ]:
lab.sql(r"""
SHOW CREATE TABLE otel_logs
""", title='OTel log bucket configuration')


`BUCKETS AUTO` enables automatic bucket calculation. It helps Doris determine the bucket count automatically, so you do not need to calculate it manually, making data loading and querying more convenient.

For bucket distribution, HASH bucketing can prune buckets efficiently when the filter matches the bucket column, but it can create hotspots when the column is heavily skewed. RANDOM bucketing is preferable for hotspot data and is used by the tables created by the OTel Doris exporter. When the data volume varies, `BUCKETS AUTO` avoids manual bucket-count calculations.

In summary, daily partitioning is the most suitable approach for most observability scenarios. If only a subset of partitions needs to be retained, dynamic partitioning is a good choice. If ease of use is the priority, automatic partitioning can be used directly.

## Next

Open [Quiz 5: Partitioning and Bucketing for Observability Data](quiz05_partition_and_bucket_choices.ipynb), then continue to [Module 6: Table Properties, Materialized Views, and Indexes](../module06-property-mv-index/lab06_properties_mv_index.ipynb).